In [ ]:
from pathlib import Path
import numpy as np

# ---- Config ----
tiles_sub_dir = Path("data/tiles_sub")
split = "train"

# ---- Count unique cells per dataset ----
dataset_counts = {}

for dataset_dir in sorted(d for d in tiles_sub_dir.iterdir() if d.is_dir()):
    mask_dir = dataset_dir / split / "masks"
    if not mask_dir.exists():
        continue

    all_labels = set()
    for fp in sorted(mask_dir.glob("*.npy")):
        mask = np.load(fp, mmap_mode="r")
        labels = np.unique(mask)
        labels = labels[labels > 0]  # ignore background
        all_labels.update(int(x) for x in labels)

    dataset_counts[dataset_dir.name] = len(all_labels)

# ---- Print results ----
if not dataset_counts:
    print(f"No datasets found with '{split}/masks/*.npy' under: {tiles_sub_dir}")
else:
    print("Cell count per dataset (unique non-zero IDs):")
    for name, count in dataset_counts.items():
        print(f"{name}: {count}")

    counts = np.array(list(dataset_counts.values()), dtype=np.float64)
    print("\nSummary across datasets:")
    print(f"Average: {counts.mean():.2f}")
    print(f"Median:  {np.median(counts):.2f}")
